# TermNorm Backend

Connect to TermNorm, sync experiments, replay pipelines, compare variants.

**Prerequisites:** TermNorm running at `http://127.0.0.1:8000`

In [ ]:
import sys
sys.path.insert(0, '..')

from api.models.backend import BackendConnection
from api.services.project_store import ProjectStore
from api.services.backend_client import BackendClient

TERMNORM_URL = "http://127.0.0.1:8000"
BACKEND_ID = "termnorm-local"

store = ProjectStore()
client = BackendClient(TERMNORM_URL)

print("Ready")

## 1. Register backend

In [ ]:
# Register (idempotent — skips if already exists)
if store.get_backend(BACKEND_ID):
    backend = store.get_backend(BACKEND_ID)
    print(f"Already registered: {backend.name} ({backend.base_url})")
else:
    backend = BackendConnection(
        id=BACKEND_ID,
        name="TermNorm Local",
        backend_type="termnorm",
        base_url=TERMNORM_URL,
    )
    store.register_backend(backend)
    print(f"Registered: {backend.name}")

print(f"Store: .promptpotter/projects/{BACKEND_ID}/")

## 2. Sync experiments from TermNorm

In [ ]:
count = await client.sync_experiments(store, BACKEND_ID)

# Update last_synced_at
from datetime import datetime, timezone
backend.last_synced_at = datetime.now(timezone.utc).isoformat()
store.update_backend(backend)

print(f"Synced {count} experiment(s)")

In [ ]:
# Peek at what we got (native TermNorm format)
experiments = store.load_sync(BACKEND_ID, "experiments.json")
for exp in experiments.get("experiments", []):
    exp_id = exp.get("experiment_id", exp.get("id", "?"))
    print(f"  {exp_id}: {exp.get('name', '')} — {exp.get('description', '')[:80]}")

## 3. Inspect a synced experiment

In [ ]:
EXPERIMENT_ID = "1_production_historical"  # adjust if needed

exp_data = store.load_sync(BACKEND_ID, f"experiments/{EXPERIMENT_ID}.json")

terms = client.extract_session_terms(exp_data)
queries = client.extract_replay_queries(exp_data)

print(f"Experiment: {exp_data.get('experiment', {}).get('name', EXPERIMENT_ID)}")
print(f"Mappings: {len(exp_data.get('mappings', []))}")
print(f"Session terms: {len(terms)}")
print(f"Replayable queries (with ground truth): {len(queries)}")
print()
print("First 5 queries:")
for q in queries[:5]:
    print(f"  {q['query'][:60]}  ->  GT: {q['ground_truth'][:50]}")

## 4. Replay (execute via TermNorm API)

Calls TermNorm's `/sessions` and `/matches` endpoints with `skip_llm_ranking=True`.

In [ ]:
import uuid
from api.models.backend import Execution, ExecutionResultItem

LIMIT = 3  # set to 0 for all queries
replay_queries = queries[:LIMIT] if LIMIT else queries

print(f"Replaying {len(replay_queries)} queries against {TERMNORM_URL}...\n")

results = await client.replay_queries(
    queries=replay_queries,
    terms=terms,
    skip_llm_ranking=True,
    delay_between=2.0,
)

# Save as execution
execution_id = uuid.uuid4().hex[:12]
successful = sum(1 for r in results if r["status"] == "success")
errors = sum(1 for r in results if r["status"] == "error")

execution = Execution(
    execution_id=execution_id,
    backend_id=BACKEND_ID,
    experiment_id=EXPERIMENT_ID,
    variant_label="LLM1-TokenMatching (no LLM2)",
    pipeline_notation="LLM1-TokenMatching",
    session_terms_count=len(terms),
    query_count=len(results),
    successful_count=successful,
    error_count=errors,
    results=[ExecutionResultItem(**r) for r in results],
)
store.save_execution(execution)

print(f"\nDone: {successful} success, {errors} errors")
print(f"Saved: execution_id={execution_id}")

In [ ]:
# Quick results table
import pandas as pd

rows = []
for r in results:
    gt = r["ground_truth"]
    pred = r.get("predicted", "")
    rows.append({
        "query": r["query"][:50],
        "predicted": pred[:50],
        "ground_truth": gt[:50],
        "correct": pred == gt,
        "latency_ms": r.get("latency_ms", 0),
    })

df = pd.DataFrame(rows)
display(df)

## 5. Compare variants

In [ ]:
from api.services.comparison import compute_comparison
import json

comparison = compute_comparison(
    results,
    metadata={
        "pipeline_notation": execution.pipeline_notation,
        "variant_label": execution.variant_label,
        "session_terms_count": execution.session_terms_count,
    },
)

m = comparison["metrics"]
c = comparison["classification"]
n = comparison["dataset"]["query_count"]

print(f"Queries: {n}")
print(f"")
print(f"Accuracy (hit@1):")
print(f"  Variant A (no LLM2): {m['hit_at_1']['a_count']}/{n} ({m['hit_at_1']['a']:.1%})")
print(f"  Variant B (full):    {m['hit_at_1']['b_count']}/{n} ({m['hit_at_1']['b']:.1%})")
print(f"")
print(f"Classification:")
print(f"  Both correct:   {c['both_correct']}")
print(f"  A-only correct: {c['a_only_correct']}  (LLM2 hurt)")
print(f"  B-only correct: {c['b_only_correct']}  (LLM2 helped)")
print(f"  Both wrong:     {c['both_wrong']}")

## 6. Browse stored executions

In [ ]:
executions = store.list_executions(BACKEND_ID)
for ex in executions:
    print(f"  {ex['execution_id']}  {ex['variant_label']}  "
          f"queries={ex['query_count']}  success={ex['successful_count']}  "
          f"{ex['created_at']}")